### **Adaptação código Aula 03:**
- SnowballC (wordsStem(), stopwords)
- Índice Inverso
- Tabela de frequência de termo por documento

In [185]:
if (!require(SnowballC)) install.packages("SnowballC")
library(SnowballC)

In [186]:
docs <- c(
  doc1 = "O gato comeu peixe",
  doc2 = "Os gatos comem peixe",
  doc3 = "O cachorro come carne"
)

In [187]:
processar_texto <- function(txt) {
    txt <- tolower(txt)
    txt <- gsub("[[:punct:]]", "", txt)
    tokens <- unlist(strsplit(txt, "\\s+"))
    tokens <- tokens[tokens != ""]
  
    stopwords_multilingual <- unique(c(
        stopwords::stopwords("pt", "snowball")))

    tokens <- tokens[
    !tokens %in% stopwords_multilingual]

    tokens <- wordStem(tokens, language = "portuguese")     # Separando RADICAL da palavra.
    return(tokens)
}

In [ ]:
postings <- list()

for (doc_id in names(docs)) {
  tokens <- processar_texto(docs[doc_id])
  tokens_unicos <- unique(tokens)
  for (tok in tokens_unicos) {
    postings[[tok]] <- c(postings[[tok]], doc_id) }
}

# Contagem de aparições por doc. de um >> PSEUDO-RADICAL <<:

print(postings[["gat"]])    # Retorna em quantos "docs"/unidades de texto 
print(postings[["peix"]])   # o PSEUDO-RADICAL "document" aparece.
print(postings[["com"]])

[1] "doc1" "doc2"
[1] "doc1" "doc2"
[1] "doc1" "doc2" "doc3"


In [190]:
preparar_consulta <- function(texto) {
  processar_texto(texto)
}

buscar_AND <- function(consulta, indice = postings) {
  termos <- preparar_consulta(consulta)
  if (length(termos) == 0) return(character(0))
  
  listas <- indice[termos]
  listas <- listas[!sapply(listas, is.null)]
  if (length(listas) == 0) return(character(0))
  Reduce(intersect, listas)
}

buscar_OR <- function(consulta, indice = postings) {
  termos <- preparar_consulta(consulta)
  if (length(termos) == 0) return(character(0))
  
  listas <- indice[termos]
  listas <- listas[!sapply(listas, is.null)]
  if (length(listas) == 0) return(character(0))
  Reduce(union, listas)
}

In [191]:
cat("Resultado_1:", buscar_AND(consulta = "peixe gato"),"\n") # 2 dos docs. possuem AMBOS os pseudo-radicais.
cat("Resultado_2:", buscar_OR(consulta = "gato comeu"))       # 3 documentos possuem ALGUM dos 2 pseudo-radicais.

Resultado_1: doc1 doc2 
Resultado_2: doc1 doc2 doc3

In [192]:
tabela_indice <- data.frame(
  Radical = names(postings),                                     # 1ª coluna (pseudo-radical listado)
  Frequencia_Docs = lengths(postings),                           # 2ª coluna (quantos docs)
  Documentos = I(sapply(postings, paste, collapse = ", "))       # 3ª coluna (d1, d2...)
)

tabela_indice <- tabela_indice[order(tabela_indice$Radical), ]
print(tabela_indice)

        Radical Frequencia_Docs       Documentos
cachorr cachorr               1             doc3
carn       carn               1             doc3
com         com               3 doc1, doc2, doc3
gat         gat               2       doc1, doc2
peix       peix               2       doc1, doc2
